# Loan Approval Predictor — EDA & Model Training

This notebook explores the loan approval dataset, cleans it, and trains a classification model to predict whether a loan application will be approved.

For the production-ready version of this pipeline (used by the deployed app), see `src/train.py` and `src/predict.py` — this notebook mirrors that logic for exploratory purposes.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

%matplotlib inline

## 2. Load the data

In [ ]:
df = pd.read_csv("../data/loan_approval_data.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Missing values

Every column has ~5% missing values. We'll impute these — but only *after* splitting into train/test, so that statistics from the test set never leak into training (this is exactly what `src/train.py` does, using a scikit-learn `Pipeline` so the imputer is fit on train data only).

In [ ]:
df.isnull().sum()

## 4. Target class balance

In [ ]:
classes_count = df["Loan_Approved"].value_counts()
plt.figure(figsize=(5, 5))
plt.pie(classes_count, labels=classes_count.index, autopct="%1.1f%%")
plt.title("Loan Approved?")
plt.show()

The classes are moderately imbalanced (~65% No / 35% Yes). We account for this by using **stratified** train/test splitting and by comparing models on **F1 score** (not just accuracy) in `src/train.py`.

## 5. Categorical feature distributions

In [ ]:
categorical_cols = df.select_dtypes(exclude=["number"]).columns.drop("Loan_Approved")

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, col in zip(axes.flatten(), categorical_cols[:6]):
    counts = df[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 6. Correlation heatmap (numeric features)

In [ ]:
numeric_df = df.select_dtypes(include="number")
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## 7. Train/test split, preprocessing, and model training

From here on, we build the same leakage-safe pipeline used by `src/train.py`: split first, then impute/encode/scale using a `Pipeline` fit only on the training data.

In [ ]:
import sys
sys.path.append("../src")
from train import build_pipeline
from preprocessing import TARGET_COLUMN, drop_id_columns

clean_df = drop_id_columns(df).dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
y = clean_df[TARGET_COLUMN].map({"Yes": 1, "No": 0})
X = clean_df.drop(columns=[TARGET_COLUMN])

categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 8. Train & compare models

We compare Logistic Regression (simple, interpretable baseline) against a Random Forest (handles non-linear relationships), and pick the better one by F1 score. See `src/train.py` for the full training + evaluation + model-saving logic — running `python src/train.py` from the project root regenerates `models/loan_model.pkl`.

In [ ]:
# See src/train.py for the full, reusable training pipeline.
# Run it from the project root with:
#     python src/train.py
print("Run `python src/train.py` from the project root to train and save the model.")

## 9. Conclusion

- Credit score, income, and DTI ratio are among the strongest predictors of loan approval.
- A Random Forest classifier outperforms plain Logistic Regression on this dataset.
- The full, production pipeline (with proper train/test isolation) lives in `src/`, and is served through the Streamlit app in `app.py`.